In [47]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [48]:
import os
os.chdir(r'C:\Users\Lenovo\Desktop\Diabetes_Classifier\notebooks')
os.chdir("..")
print(os.getcwd())

C:\Users\Lenovo\Desktop\Diabetes_Classifier


In [49]:
from modeling.XGBoost import XGBoost
from modeling.RandomForest import RandomForest
from modeling.KNN import KNN
from modeling.MLP import MLP
from modeling.model import ModelMetrics
from sklearn.linear_model import LogisticRegression
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score,accuracy_score

In [50]:
PATH="data/preprocessed/final.csv"
df=pd.read_csv(PATH)
df.head()
df.columns = df.columns.str.strip()

In [51]:


y=df['diagnosis']
X=df.drop(['diagnosis'],axis=1)
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)


In [52]:

xgb=XGBoost()
xgb_params={'n_estimators': 7300, 'max_depth': 9,'learning_rate': 0.042826934350783406, 'gamma': 0.8864325835995647, 'min_child_weight': 1, 'reg_alpha': 0.00019361810915135758, 'reg_lambda': 3.6728295355053825e-06, 'subsample': 0.6459234341228212, 'colsample_bytree': 0.2968183856347904, 'tree_method': 'hist', 'n_jobs': -1}
xgb.set_params(xgb_params)
pd.set_option('display.max_rows',None)

# OOF
# X=xgb.oof(X,y)

#Optuna
# params=xgb.hyperparameter_tuning(X,y,30)
# xgb.set_params(params)

# Train and Test XGB
metrics=xgb.evaluation(X_train,y_train,5)
metrics.out()

# xgb.feature_importance(X.columns)

# preds=xgb.predict(X_test)
# score=roc_auc_score(y_test,preds)
# print(score)



Accuracy: 0.8192448233861146
Precision: 0.8122945084796225
F1: 0.807752977552823
Recall: 0.8052776186076726
ROC-AUC: 0.9103467849377775


In [53]:
rf=RandomForest()
rf_params={'n_estimators': 600, 'max_depth': 41, 'min_samples_split': 17, 'min_samples_leaf': 6, 'max_features': 'log2', 'bootstrap': True, 'criterion': 'log_loss', 'class_weight': None, 'n_jobs': -1, 'random_state': 42}
rf.set_params(rf_params)

metrics=rf.evaluation(X_train,y_train,5)
metrics.out()
# rf_params=rf.hyperparameter_tuning(X_train,y_train,20)
# rf.feature_importance(X_train.columns)


Accuracy: 0.8263093788063338
Precision: 0.8198882268579716
F1: 0.8152958605258533
Recall: 0.8125560538356785
ROC-AUC: 0.9161125065931696


In [54]:
mlp=MLP()
mlp_params={'hidden_layer_sizes': (128, 64, 32), 'activation': 'relu', 'solver': 'adam', 'alpha': 3.833957124267705e-05, 'learning_rate_init': 0.001020282896183049, 'batch_size': 32, 'max_iter': 500, 'early_stopping': True, 'random_state': 42}
mlp.set_params(mlp_params)

X_train_scaled,X_test_scaled=mlp.scaling(X_train,X_test)
metrics=mlp.evaluation(X_train_scaled,y_train,5)
metrics.out()

# params=mlp.hyperparameter_tuning(X_train_scaled,y_train_scaled,20)
# mlp.set_params(params)


Accuracy: 0.8243605359317906
Precision: 0.8181108825360859
F1: 0.8137071913776601
Recall: 0.8127963871645469
ROC-AUC: 0.912770024707048


In [55]:
knn=KNN()
knn_params={'n_neighbors': 50, 'weights': 'uniform', 'metric': 'manhattan', 'p': 3, 'algorithm': 'auto', 'leaf_size': 34, 'n_jobs': -1}
knn.set_params(knn_params)

metrics=knn.evaluation(X_train_scaled,y_train,5)
metrics.out()

# parameters=knn.hyperparameter_tuning(X_train_scaled,y_train_scaled,20)

Accuracy: 0.8138855054811206
Precision: 0.8125853213564602
F1: 0.7976147584949301
Recall: 0.7906617741954994
ROC-AUC: 0.9095757297751735


In [56]:
ensemble=[xgb,rf,mlp,knn]
X_train_copy=X_train_scaled.copy()
X_test_copy=X_test_scaled.copy()
for model in ensemble:
    X_train_scaled[f'OOF_{model}'],X_test_scaled[f'OOF_{model}']=model.oof(X_train,y_train,X_test,model,10)

<modeling.XGBoost.XGBoost object at 0x00000241AF2E79D0> fold 1/10 done
<modeling.XGBoost.XGBoost object at 0x00000241AF2E79D0> fold 2/10 done
<modeling.XGBoost.XGBoost object at 0x00000241AF2E79D0> fold 3/10 done
<modeling.XGBoost.XGBoost object at 0x00000241AF2E79D0> fold 4/10 done
<modeling.XGBoost.XGBoost object at 0x00000241AF2E79D0> fold 5/10 done
<modeling.XGBoost.XGBoost object at 0x00000241AF2E79D0> fold 6/10 done
<modeling.XGBoost.XGBoost object at 0x00000241AF2E79D0> fold 7/10 done
<modeling.XGBoost.XGBoost object at 0x00000241AF2E79D0> fold 8/10 done
<modeling.XGBoost.XGBoost object at 0x00000241AF2E79D0> fold 9/10 done
<modeling.XGBoost.XGBoost object at 0x00000241AF2E79D0> fold 10/10 done
<modeling.RandomForest.RandomForest object at 0x00000241AFF4EFD0> fold 1/10 done
<modeling.RandomForest.RandomForest object at 0x00000241AFF4EFD0> fold 2/10 done
<modeling.RandomForest.RandomForest object at 0x00000241AFF4EFD0> fold 3/10 done
<modeling.RandomForest.RandomForest object at 

In [57]:

print(X_train_scaled.isna().sum())

age                                                                      0
gender                                                                   0
bmi                                                                      0
chol                                                                     0
tg                                                                       0
hdl                                                                      0
ldl                                                                      0
cr                                                                       0
bun                                                                      0
lipids                                                                   0
Age_x_BMI                                                                0
HDL_x_LDL                                                                0
BMI/LDL                                                                  0
BMI/HDL+LDL              

In [58]:

X_train_scaled.tail(2)

,age,gender,bmi,chol,tg,hdl,ldl,cr,bun,lipids,...,Cluster 1,Cluster 2,Cluster 3,Cluster 4,Cluster 5,anomaly_score,OOF_<modeling.XGBoost.XGBoost object at 0x00000241AF2E79D0>,OOF_<modeling.RandomForest.RandomForest object at 0x00000241AFF4EFD0>,OOF_<modeling.MLP.MLP object at 0x00000241AF5A9F90>,OOF_<modeling.KNN.KNN object at 0x00000241AFEA4190>
4103,1.431108,0.760819,-0.373476,-0.861977,0.230751,3.275061,2.080145,0.303456,-0.529966,1.358916,...,-2.368231,1.404717,0.636200,1.714660,1.147138,-0.287017,0.999567,1.000000,0.997310,0.66
4104,-0.638830,-1.314372,-0.604960,2.446873,-0.625225,-0.033274,2.302451,-0.596756,-0.357737,-0.803993,...,-0.296889,0.392921,0.760038,1.316515,-0.011318,-0.287017,0.051255,0.264314,0.229554,0.06


In [59]:
meta_x_model=LogisticRegression(    penalty='l2',
    C=0.1,
    solver='lbfgs',
    max_iter=5000,
    class_weight='balanced',   # or None
    random_state=42,
    n_jobs=-1)

In [60]:
meta_x_model.fit(X_train_scaled,y_train)
lr_proba=meta_x_model.predict_proba(X_test_scaled)
preds=meta_x_model.predict(X_test_scaled)


c:\Users\Lenovo\anaconda3\envs\Diabetes\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\Lenovo\anaconda3\envs\Diabetes\Lib\site-packages\sklearn\linear_model\_logistic.py:1457: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


In [61]:
from sklearn.metrics import f1_score,recall_score,precision_score,balanced_accuracy_score
def measure(y_pred: np.ndarray, y_proba: np.ndarray, y_real: np.ndarray) -> dict:
    metrics = {}
    metrics['accuracy'] = accuracy_score(y_real, y_pred)
    metrics['balanced']=balanced_accuracy_score(y_real,y_pred)
    metrics['precision'] = precision_score(y_real, y_pred, average='macro')
    metrics['f1'] = f1_score(y_real, y_pred, average='macro')
    metrics['recall'] = recall_score(y_real, y_pred, average='macro')
    metrics['roc_auc'] = roc_auc_score(y_real, y_proba[:,1], average='macro')
    return metrics

    

In [62]:
def compare(base: dict, meta: dict):
    for key in base:
        diff = meta[key] - base[key]
        symbol = "↑" if diff > 0 else ("↓" if diff < 0 else "=")
        print(f"{key}: base={base[key]:.4f} meta={meta[key]:.4f} {symbol} ({diff:+.4f})")


In [63]:
logistic_regression_metrics=measure(preds,lr_proba,y_test)
print(logistic_regression_metrics)

{'accuracy': 0.8227848101265823, 'balanced': 0.8201242102411702, 'precision': 0.8132411067193676, 'f1': 0.8160150405543743, 'recall': 0.8201242102411702, 'roc_auc': 0.9113358775584999}


In [64]:
for model in ensemble:
    if model.__class__.__name__ in ["XGBoost", "RandomForest"]:
        model.train(X_train,y_train)
    else:
        model.train(X_train_copy,y_train)

In [65]:
for model in ensemble:
    print(f'Model:{model}')
    if model.__class__.__name__ in ["XGBoost", "RandomForest"]:
        predictions=model.predict(X_test)
        proba=model.predict_proba(X_test)
        vals=measure(predictions,proba,y_test)
        compare(vals,logistic_regression_metrics)
    else:
        predictions=model.predict(X_test_copy)
        proba=model.predict_proba(X_test_copy)
        vals=measure(predictions,proba,y_test)
        compare(vals,logistic_regression_metrics)

Model:<modeling.XGBoost.XGBoost object at 0x00000241AF2E79D0>
accuracy: base=0.8199 meta=0.8228 ↑ (+0.0029)
balanced: base=0.8061 meta=0.8201 ↑ (+0.0140)
precision: base=0.8125 meta=0.8132 ↑ (+0.0007)
f1: base=0.8089 meta=0.8160 ↑ (+0.0071)
recall: base=0.8061 meta=0.8201 ↑ (+0.0140)
roc_auc: base=0.9084 meta=0.9113 ↑ (+0.0030)
Model:<modeling.RandomForest.RandomForest object at 0x00000241AFF4EFD0>
accuracy: base=0.8179 meta=0.8228 ↑ (+0.0049)
balanced: base=0.8031 meta=0.8201 ↑ (+0.0170)
precision: base=0.8109 meta=0.8132 ↑ (+0.0024)
f1: base=0.8064 meta=0.8160 ↑ (+0.0096)
recall: base=0.8031 meta=0.8201 ↑ (+0.0170)
roc_auc: base=0.9126 meta=0.9113 ↓ (-0.0013)
Model:<modeling.MLP.MLP object at 0x00000241AF5A9F90>
accuracy: base=0.8325 meta=0.8228 ↓ (-0.0097)
balanced: base=0.8214 meta=0.8201 ↓ (-0.0013)
precision: base=0.8251 meta=0.8132 ↓ (-0.0118)
f1: base=0.8231 meta=0.8160 ↓ (-0.0071)
recall: base=0.8214 meta=0.8201 ↓ (-0.0013)
roc_auc: base=0.9107 meta=0.9113 ↑ (+0.0006)
Model:<m

In [66]:
X_train_scaled.iloc[:,-4:].corr()

,OOF_<modeling.XGBoost.XGBoost object at 0x00000241AF2E79D0>,OOF_<modeling.RandomForest.RandomForest object at 0x00000241AFF4EFD0>,OOF_<modeling.MLP.MLP object at 0x00000241AF5A9F90>,OOF_<modeling.KNN.KNN object at 0x00000241AFEA4190>
OOF_<modeling.XGBoost.XGBoost object at 0x00000241AF2E79D0>,1.000000,0.964931,0.895582,0.864886
OOF_<modeling.RandomForest.RandomForest object at 0x00000241AFF4EFD0>,0.964931,1.000000,0.929359,0.898895
OOF_<modeling.MLP.MLP object at 0x00000241AF5A9F90>,0.895582,0.929359,1.000000,0.910940
OOF_<modeling.KNN.KNN object at 0x00000241AFEA4190>,0.864886,0.898895,0.910940,1.000000


In [67]:
coefs = meta_x_model.coef_[0]  # flatten the (1, n_features) row
feature_cols=X_train_scaled.columns
for name, c in zip(feature_cols, coefs):
    print(f"{name}: {c:.4f}")
print(meta_x_model.classes_)

age: 0.4427
gender: -0.0142
bmi: 0.2235
chol: -0.2087
tg: 0.0853
hdl: 0.2515
ldl: 0.1991
cr: -0.4122
bun: 0.0390
lipids: -0.1839
Age_x_BMI: 0.2298
HDL_x_LDL: 0.4832
BMI/LDL: -0.2189
BMI/HDL+LDL: 0.2206
bun_x_cr: 0.2438
chol/ldl: 0.1940
cluster_labels: -0.0497
Cluster 0: -0.1560
Cluster 1: -0.0235
Cluster 2: 0.6280
Cluster 3: 0.0664
Cluster 4: -0.1184
Cluster 5: -0.0662
anomaly_score: 0.1539
OOF_<modeling.XGBoost.XGBoost object at 0x00000241AF2E79D0>: 0.9762
OOF_<modeling.RandomForest.RandomForest object at 0x00000241AFF4EFD0>: 1.5112
OOF_<modeling.MLP.MLP object at 0x00000241AF5A9F90>: -0.3394
OOF_<modeling.KNN.KNN object at 0x00000241AFEA4190>: 0.2496
[0 1]
